Task 1:
1. Difference between Love and love in NLP?
   
   In NLP, "Love" and "love" are treated as two completely different tokens by default, because most NLP models and tokenizers are case-sensitive.

   A word appearing at the start of a sentence is usually capitalized (e.g., "Love is beautiful"), while the same word mid-sentence is lowercase (e.g., "I love music"). Without lowercasing, the model would consider these as different features, doubling the vocabulary size unnecessarily.
  
   This is why lowercasing is one of the very first preprocessing steps — it maps all case variants to a single, unified token.

   Exception: In Named Entity Recognition (NER), capitalization is a meaningful signal. "Apple" (a company) vs "apple" (a fruit) should NOT be lowercased blindly. In such cases, case-sensitive preprocessing is preferred.

2.  What happens if stopwords are not removed?

   Stopwords are high-frequency, low-information words like "is", "the", "a", "in", "and", "of". If they are not removed:

  - Vocabulary explosion: The feature space grows unnecessarily large, slowing down model training.

  - Noise in frequency analysis: Words like "the" and "is" dominate word counts, masking actually important words.

   - Poor model performance: ML models may focus on these irrelevant tokens, reducing accuracy on tasks like classification or clustering.

- Misleading similarity: Two documents on completely different topics may appear similar just because they share many stopwords.

- Wasted memory & compute: Processing and storing tokens with zero semantic value wastes resources.

3. Two real-world scenarios where removing stopwords can be HARMFUL

Scenario 1 — Sentiment Analysis with Negations:

"I am happy" vs "I am NOT happy"

The word "not" is a standard stopword. Removing it transforms both sentences into just "happy", completely inverting the sentiment of the second sentence. Any sentiment classifier trained on such data would give incorrect results, potentially flagging a negative review as positive.

 Scenario 2 — Question Answering & Intent Detection:

"Can you help?" vs "Can you not help?" "Who is the manager?" vs "Who is not the manager?"

Function words and auxiliary verbs like "not", "is", "can", "will" define the intent and structure of the question. Stripping them out makes it impossible to distinguish between a request and its negation, causing the system to give wrong responses.

4. Difference between Stemming and Lemmatization

### **Stemming vs Lemmatization**

| Feature | Stemming | Lemmatization |
|--------|--------|--------------|
| Method | Rule-based | Dictionary-based |
| Output | May be incorrect | Meaningful |
| Speed | Fast | Slower |
| Accuracy | Low | High |
| Example | studies → studi | better → good |
| Algorithm | Porter Stemmer | WordNetLemmetizer(NLTK)
|Use Case | Search Engines | NLP Tasks


To sum up: Stemming is fast but crude and may produce non-words. Lemmatization is slower but always returns a linguistically valid base form. For production NLP systems, lemmatization is preferred.


Tak 2: Build Advanced Preprocessing Function

In [20]:
import re

def preprocess_text(text):
    # --- ERROR HANDLING: Check if input is empty or null ---
    if not text or not isinstance(text, str) or text.strip() == "":
        return [], ""
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Lowercase
    text = text.lower()
    # Handle repeated characters
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    # Remove special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    tokens = text.split()
    # Remove short words except 'no', 'not'
    tokens = [w for w in tokens if len(w) > 2 or w in ['no', 'not']]
    cleaned_sentence = " ".join(tokens)
    return tokens, cleaned_sentence

    message=["I have 2 dogs","This is   good","soooo goooood!!!","WOW!!! This is GREAT!!!","Visit http://example.com now"  ]
tokens=[]
cleaned_sentence=[]
for text in message:
    token,sentence=preprocess_text(text)
    tokens.append(token)
    cleaned_sentence.append(sentence)
print(f"Tokens: {tokens}")
print(f"cleaned_sentence: {cleaned_sentence}")

Tokens: [['have', 'dogs'], ['this', 'good'], ['god'], ['wow', 'this', 'great'], ['visit', 'now']]
cleaned_sentence: ['have dogs', 'this good', 'god', 'wow this great', 'visit now']


Task 3: Stress Testing
Test your function on at least 10 diverse sentences, including:

● Emojis


● Numbers


● Slang


● Repeated characters


● URLs


● Mixed case

In [21]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍🔥",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

# Run the stress test
for original in test_sentences:
    token_list, final_sentence = preprocess_text(original)

    print(f"Original Text:     {original}")
    print(f"Cleaned Tokens:    {token_list}")
    print(f"Cleaned Sentence:  {final_sentence}")
    print("-" * 40)

Original Text:     Get 100% FREE access now!!!
Cleaned Tokens:    ['get', 'free', 'access', 'now']
Cleaned Sentence:  get free access now
----------------------------------------
Original Text:     I absolutely looooved this product 😍🔥
Cleaned Tokens:    ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence:  absolutely loved this product
----------------------------------------
Original Text:     Worst service ever... 0/10
Cleaned Tokens:    ['worst', 'service', 'ever']
Cleaned Sentence:  worst service ever
----------------------------------------
Original Text:     Call me at 9876543210
Cleaned Tokens:    ['call']
Cleaned Sentence:  call
----------------------------------------
Original Text:     This is THE best course!!!
Cleaned Tokens:    ['this', 'the', 'best', 'course']
Cleaned Sentence:  this the best course
----------------------------------------
Original Text:     Visit https://openai.com now!
Cleaned Tokens:    ['visit', 'now']
Cleaned Sentence:  visit now
-----------

Task 4:  Token Analytics
For each processed sentence, compute:
1. Total number of tokens
2. Number of unique tokens
3. Average token length

Analysis Questions

● Which sentence had the most noise?

● Which sentence retained the most meaningful tokens after cleaning?

In [8]:
def token_analytics(tokens):
    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))

    avg_length = (
        sum(len(word) for word in tokens) / total_tokens
        if total_tokens > 0 else 0
    )

    return total_tokens, unique_tokens, round(avg_length, 2)


print("Output\n")

for sentence in test_sentences:
    tokens, clean = preprocess_text(sentence)
    total, unique, avg = token_analytics(tokens)

    print("Sentence        :", sentence)
    print("Total Tokens    :", total)
    print("Unique Tokens   :", unique)
    print("Avg Token Length:", avg)
    print("-" * 50)

Output

Sentence        : Get 100% FREE access now!!!
Total Tokens    : 4
Unique Tokens   : 4
Avg Token Length: 4.0
--------------------------------------------------
Sentence        : I absolutely looooved this product 😍😍
Total Tokens    : 4
Unique Tokens   : 4
Avg Token Length: 6.5
--------------------------------------------------
Sentence        : Worst service ever... 0/10
Total Tokens    : 3
Unique Tokens   : 3
Avg Token Length: 5.33
--------------------------------------------------
Sentence        : Call me at 9876543210
Total Tokens    : 1
Unique Tokens   : 1
Avg Token Length: 4.0
--------------------------------------------------
Sentence        : This is THE best course!!!
Total Tokens    : 4
Unique Tokens   : 4
Avg Token Length: 4.25
--------------------------------------------------
Sentence        : Visit https://openai.com now!
Total Tokens    : 2
Unique Tokens   : 2
Avg Token Length: 4.0
--------------------------------------------------
Sentence        : Nooooo this is

1. Which sentence had the most noise?

The sentence with the most noise was: "Visit https://openai.com now!" (and closely followed by "Call me at 9876543210").
In NLP, "noise" refers to characters or words that don't provide semantic value for general analysis.

The URL "https://openai.com" and the exclamation mark were entirely removed. In the phone number example, the entire string of digits was wiped out, leaving only one word ("call") behind. These sentences lost the highest percentage of their original "content" because they were composed almost entirely of characters your rules (correctly) identified as noise.

2. Which sentence retained the most meaningful tokens?

The sentence that retained the most meaningful tokens was: "I am not happy with this". Even though it’s a simple sentence, almost every word was preserved because they met your criteria (longer than 2 characters or in your "not" exception list).
It kept "not," "happy," "with," and "this." These four tokens provide a very clear picture of the user's intent and sentiment. Unlike the "looooved" example (which turned "looooved" into "loved"—a 5-letter word), this sentence didn't require heavy regex reconstruction to remain readable.

Task 5: Frequency Analysis

In [15]:
from collections import Counter

all_tokens = []
for s in test_sentences:
    tokens, _ = preprocess_text(s)
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10:", counter.most_common(10))
print("Least 5:", counter.most_common()[-5:])

Top 10: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Least 5: [('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


Task 6: Build Full Pipeline

In [16]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.append(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

text_list=["This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it", ]
full_pipeline(text_list)

{'tokens': [['this', 'the', 'best', 'course'],
  ['visit', 'now'],
  ['no', 'this', 'bad'],
  ['got']],
 'clean_sentences': ['this the best course',
  'visit now',
  'no this bad',
  'got']}

Task 7: Error Handling : Ensure your solution handles:

 ● Empty string


 ● Only emojis


 ● Only numbers

In [17]:
error_test_cases = ["", "😊😍🚀", "9876543210", "   "]

for text in error_test_cases:
    token, sentence = preprocess_text(text)

    # These will naturally return [], "" because your regex removes all emojis/numbers
    print(f"Input: '{text}'")
    print(f"Tokens: {token} | Sentence: '{sentence}'")
    print("-" * 30)

Input: ''
Tokens: [] | Sentence: ''
------------------------------
Input: '😊😍🚀'
Tokens: [] | Sentence: ''
------------------------------
Input: '9876543210'
Tokens: [] | Sentence: ''
------------------------------
Input: '   '
Tokens: [] | Sentence: ''
------------------------------
